# Notebook 06 — Baseline Inference Smoke Test (v1.0.0)

Purpose:
- Load the active baseline model from `models/model_registry.json`
- Build the exact inference feature schema using lookup enrichment
- Run single-request + batch sanity predictions

This notebook is intentionally **not** wiring the FastAPI endpoint yet.

In [1]:
from __future__ import annotations

import json
import datetime as dt
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

pd.set_option('display.max_columns', 200)

def find_ml_root(start: Path) -> Path:
    current = start.resolve()
    for _ in range(6):
        if (current / 'models' / 'model_registry.json').exists():
            return current
        current = current.parent
    raise FileNotFoundError('Could not find ml-service root containing models/model_registry.json')

ML_ROOT = find_ml_root(Path.cwd())
REGISTRY_PATH = ML_ROOT / 'models' / 'model_registry.json'
PROCESSED_PATH = ML_ROOT / 'data' / 'processed' / 'processed_data.csv'
LOOKUP_PATH = ML_ROOT / 'data' / 'lookups' / 'car_specs_lookup_full_cleaned.csv'

print('ML_ROOT:', ML_ROOT)
print('REGISTRY_PATH:', REGISTRY_PATH)
print('PROCESSED_PATH:', PROCESSED_PATH)
print('LOOKUP_PATH:', LOOKUP_PATH)

ML_ROOT: /home/mo-seif/Documents/GP/ml-service
REGISTRY_PATH: /home/mo-seif/Documents/GP/ml-service/models/model_registry.json
PROCESSED_PATH: /home/mo-seif/Documents/GP/ml-service/data/processed/processed_data.csv
LOOKUP_PATH: /home/mo-seif/Documents/GP/ml-service/data/lookups/car_specs_lookup_full_cleaned.csv


In [2]:
with open(REGISTRY_PATH, 'r') as f:
    registry = json.load(f)

active_version = registry['active_version']
active_model = registry['active_model']
active_model_path = ML_ROOT / active_model['pkl_path']
active_preprocessor_meta_path = ML_ROOT / registry['versions'][active_version]['preprocessor']['meta_path']

print('Active version:', active_version)
print('Active model:', active_model['model_type'], 'rank', active_model['rank'])
print('Model artifact:', active_model_path)
print('Preprocessor meta:', active_preprocessor_meta_path)

KeyError: 'active_model'

In [ ]:
model = joblib.load(active_model_path)
print(model)

In [ ]:
with open(active_preprocessor_meta_path, 'r') as f:
    pre_meta = json.load(f)

numeric_features = list(pre_meta['numeric_features'])
categorical_features = list(pre_meta['categorical_features'])
required_features = numeric_features + categorical_features

print('Numeric features:', numeric_features)
print('Categorical features:', categorical_features)
print('Total required:', len(required_features))

In [ ]:
processed = pd.read_csv(PROCESSED_PATH)
lookup = pd.read_csv(LOOKUP_PATH)

print('processed shape:', processed.shape)
print('lookup shape:', lookup.shape)

missing_required = [c for c in required_features if c not in processed.columns]
if missing_required:
    raise ValueError(f'processed_data.csv is missing required features: {missing_required}')

reference_year = int(pd.to_numeric(processed['year'], errors='coerce').dropna().max())
print('reference_year (from processed data):', reference_year)

In [ ]:
def _mode_or_unknown(series: pd.Series) -> str:
    series = series.dropna().astype(str)
    if series.empty:
        return 'Unknown'
    return series.mode().iloc[0]

def build_features_from_request(
    request: dict,
    lookup_df: pd.DataFrame,
    processed_df: pd.DataFrame,
    *,
    numeric_cols: list[str],
    categorical_cols: list[str],
    reference_year: int,
) -> pd.DataFrame:
    # Map API fields → training feature names
    make = str(request['brand']).strip()
    model_name = str(request['model']).strip()
    year = int(request['year'])
    mileage_km = float(request['mileage_km'])

    base = pd.DataFrame([{
        'make': make,
        'model': model_name,
        'year': year,
        'mileage_km': mileage_km,
        'transmission': str(request['transmission']).strip(),
        'fuel': str(request['fuel']).strip(),
        'location': str(request['location']).strip(),
    }])

    # Enrich from lookup on (make, model, year)
    lk = lookup_df.copy()
    lk['make'] = lk['make'].astype(str).str.strip()
    lk['model'] = lk['model'].astype(str).str.strip()
    lk['year'] = pd.to_numeric(lk['year'], errors='coerce').astype('Int64')

    base['year'] = pd.to_numeric(base['year'], errors='coerce').astype('Int64')
    enriched = base.merge(lk, on=['make', 'model', 'year'], how='left', suffixes=('', '_lk'))

    # Engineer
    year_int = int(enriched.loc[0, 'year'])
    car_age = max(0, reference_year - year_int)
    enriched['car_age'] = float(car_age)
    enriched['mileage_per_year'] = float(mileage_km / max(car_age, 1))

    # Ensure required columns exist
    for col in numeric_cols + categorical_cols:
        if col not in enriched.columns:
            enriched[col] = np.nan

    # Fill missing values using processed-data priors
    priors_num = processed_df[numeric_cols].apply(pd.to_numeric, errors='coerce').median(numeric_only=True)
    for col in numeric_cols:
        enriched[col] = pd.to_numeric(enriched[col], errors='coerce')
        if pd.isna(enriched.loc[0, col]):
            enriched.loc[0, col] = float(priors_num[col]) if col in priors_num.index else 0.0

    for col in categorical_cols:
        if pd.isna(enriched.loc[0, col]) or str(enriched.loc[0, col]).strip() == '':
            enriched.loc[0, col] = _mode_or_unknown(processed_df[col]) if col in processed_df.columns else 'Unknown'
        enriched[col] = enriched[col].astype(str)

    # Return in the exact required feature order
    return enriched[numeric_cols + categorical_cols].copy()

In [ ]:
example_request = {
    'brand': 'Toyota',
    'model': 'Corolla',
    'year': 2018,
    'mileage_km': 85000,
    'transmission': 'Automatic',
    'fuel': 'petrol',
    'location': 'Cairo',
}

X_new = build_features_from_request(
    example_request,
    lookup,
    processed,
    numeric_cols=numeric_features,
    categorical_cols=categorical_features,
    reference_year=reference_year,
)

X_new

In [ ]:
pred_raw = float(model.predict(X_new)[0])

# Heuristic: if prediction looks like log-price, convert to EGP
pred_egp = float(np.exp(pred_raw)) if pred_raw < 50 else pred_raw

neg_low = pred_egp * 0.85
neg_high = pred_egp * 1.15

print('pred_raw:', pred_raw)
print('fair_price_egp:', round(pred_egp, 2))
print('negotiation_range:', round(neg_low, 2), '-', round(neg_high, 2))
print('predicted_at:', dt.datetime.utcnow().isoformat() + 'Z')
print('model_version:', active_version)

In [ ]:
# Sanity check: compare prediction to similar cars in processed data
make = X_new.loc[0, 'make']
model_name = X_new.loc[0, 'model']
subset = processed[(processed['make'] == make) & (processed['model'] == model_name)]

if subset.empty:
    print('No exact make/model matches found in processed_data.csv for distribution check.')
else:
    prices = pd.to_numeric(subset['price_egp'], errors='coerce').dropna()
    q05, q50, q95 = prices.quantile([0.05, 0.5, 0.95]).tolist()
    print(f'n_similar={len(prices)} | p05={q05:,.0f} | median={q50:,.0f} | p95={q95:,.0f}')
    print('prediction:', f'{pred_egp:,.0f}')
    if pred_egp < q05 or pred_egp > q95:
        print('⚠️  Prediction is outside [p05, p95] for same make/model (may still be OK if trim differs).')

In [ ]:
# Batch smoke test: predictions on a sample of processed rows
X_sample = processed[required_features].sample(50, random_state=42)
preds_raw = model.predict(X_sample)
preds_egp = np.exp(preds_raw) if np.nanmedian(preds_raw) < 50 else preds_raw

print('preds_raw median:', float(np.nanmedian(preds_raw)))
print('preds_egp describe:')
print(pd.Series(preds_egp).describe())

if np.any(~np.isfinite(preds_egp)):
    raise ValueError('Non-finite predictions detected in batch smoke test')

## Optional: make baseline metadata portable

The baseline per-model metadata JSON currently contains absolute paths (machine-specific).
The registry already uses relative paths; this cell rewrites metadata JSON artifacts to relative paths **only when the absolute path lives under ML_ROOT**.

In [ ]:
from glob import glob

meta_dir = ML_ROOT / 'models' / 'metadata'
meta_files = sorted(meta_dir.glob('Baseline*.json'))

def _rel_if_under_root(p: str) -> str:
    try:
        path = Path(p).resolve()
    except Exception:
        return p
    try:
        rel = path.relative_to(ML_ROOT)
        return str(rel.as_posix())
    except ValueError:
        return p

rewritten = []
for mf in meta_files:
    with open(mf, 'r') as f:
        data = json.load(f)

    artifacts = data.get('artifacts', {})
    before = dict(artifacts)
    for k in ['model_pkl', 'preprocessor_pkl']:
        if k in artifacts and isinstance(artifacts[k], str):
            artifacts[k] = _rel_if_under_root(artifacts[k])

    data['artifacts'] = artifacts
    if artifacts != before:
        with open(mf, 'w') as f:
            json.dump(data, f, indent=2)
        rewritten.append(mf.name)

print('Rewritten metadata files:', rewritten)